# M3: Entrenamiento de Keypoints de Ruedas / Conteo de Ejes (YOLO11-pose) y Validación M1 + OCR
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio Final de Hito: Sistema Inteligente de Control de Peaje

Este notebook cubre dos requisitos clave de la rúbrica y del informe técnico:
1. **Entrenamiento de M3 (Keypoints sobre ruedas → Conteo de ejes):**
   - Soporta tanto el esquema de **1 keypoint por rueda (centro de rueda visible)** como el esquema geométrico de **5 keypoints por llanta (`centro`, `contacto`, `superior`, `izquierda`, `derecha`)** visto en clase.
   - Incluye auto-etiquetado inicial con `YOLOE-seg`, descarga del dataset etiquetado en **Roboflow**, entrenamiento de `yolo11s-pose.pt`, cálculo de `mAP50-pose` y evaluación del **Error Absoluto Medio (MAE) y Exactitud en el Conteo de Ejes**.
2. **Validación de M1 (Detección de Placas + EasyOCR) sobre las capturas reales de Bolivia:**
   - Evalúa la precisión de lectura y normalización sobre las placas bolivianas tomadas para el laboratorio (`3034DUS`, `1620AEN`, `4222TLN`, `5314PFC`, `1152SFU`, `4694INB`, `1857NTE`, `4763FSS`, `1651FUL`, `4454FGX`, `4695ELB`, `1605KLX`, `1367ZTG`, `1205HGL`, `1318IZH`, `1506UAX`) midiendo Character Error Rate (CER) y Exactitud de Placa Completa.

## 1. Instalación de dependencias

In [ ]:
!nvidia-smi -L || echo "ATENCIÓN: Active la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno"
!pip -q install -U ultralytics roboflow easyocr pandas matplotlib seaborn
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'

## 2. Configuración y utilidades geométricas para llantas y ejes

In [ ]:
import os, glob, time, random, shutil, json, re, cv2, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo activo:', DEVICE)

KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']
FLIP_IDX = [0, 1, 2, 4, 3]
RUTA_PESOS_M3 = '/content/runs/pose/m3_llantas/weights/best.pt'

UMBRAL_CONTENCION = 0.70
UMBRAL_CERCANA = 0.60
SEPARACION_EJE = 0.55

def contenida(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area = max(1e-6, (a[2] - a[0]) * (a[3] - a[1]))
    return inter / area

def limpiar_llantas(llantas):
    orden = sorted(llantas, key=lambda l: -(l['box'][2] - l['box'][0]) * (l['box'][3] - l['box'][1]))
    res = []
    for l in orden:
        if all(contenida(l['box'], r['box']) < UMBRAL_CONTENCION for r in res):
            res.append(l)
    return res

def estimar_ejes_desde_llantas(llantas, categoria='auto'):
    """
    Devuelve el número de ejes y la lista de keypoints centrales [[x, y, conf], ...]
    conforme al diagrama M3 de la guía del laboratorio (página 5).
    """
    llantas = limpiar_llantas(llantas)
    if not llantas:
        ejes_defecto = 2
        return ejes_defecto, []

    alturas = [max(1.0, l['box'][3] - l['box'][1]) for l in llantas]
    hmax = max(alturas)
    cercanas = [l for l, h in zip(llantas, alturas) if h >= UMBRAL_CERCANA * hmax]
    cercanas = sorted(cercanas, key=lambda l: l['centro'][0])

    xs = [l['centro'][0] for l in cercanas]
    diam = float(np.median([max(1.0, l['box'][3] - l['box'][1]) for l in cercanas]))
    ejes = 1 + sum(1 for p, q in zip(xs, xs[1:]) if (q - p) > SEPARACION_EJE * diam)
    if ejes < 2:
        ejes = 2
    keypoints_salida = [[round(float(l['centro'][0]), 1), round(float(l['centro'][1]), 1), round(float(l['conf']), 3)] for l in cercanas]
    return int(ejes), keypoints_salida

## 3. Descarga del Dataset de Keypoints de Ruedas desde Roboflow (o auto-etiquetado con YOLOE)

In [ ]:
from google.colab import userdata

def obtener_secreto(nombre, por_defecto=''):
    try:
        val = userdata.get(nombre)
        return val if val else os.environ.get(nombre, por_defecto)
    except Exception:
        return os.environ.get(nombre, por_defecto)

RF_API_KEY = obtener_secreto('ROBOFLOW_API_KEY')
RF_WORKSPACE = obtener_secreto('ROBOFLOW_WORKSPACE', 'peaje-iam811')
RF_PROJECT_M3 = obtener_secreto('ROBOFLOW_PROJECT_M3', 'ruedas-ejes-pose')
RF_VERSION_M3 = int(obtener_secreto('ROBOFLOW_VERSION_M3', '1'))

DATA_YAML_M3 = None
if RF_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=RF_API_KEY)
    ds_m3 = rf.workspace(RF_WORKSPACE).project(RF_PROJECT_M3).version(RF_VERSION_M3).download('yolov11')
    DATA_YAML_M3 = os.path.join(ds_m3.location, 'data.yaml')
    print('Dataset M3 descargado desde Roboflow en:', DATA_YAML_M3)
else:
    print('Configure ROBOFLOW_API_KEY en Colab Secrets o ejecute la celda de auto-etiquetado de keypoints_llantas_yolo_detectron2_v3.ipynb.')

## 4. Entrenamiento de M3 (`yolo11s-pose.pt`) y registro de métricas (`mAP50-pose` y error en conteo de ejes)

In [ ]:
assert DATA_YAML_M3 and os.path.exists(DATA_YAML_M3), 'Defina DATA_YAML_M3 antes de entrenar.'

modelo_m3 = YOLO('yolo11s-pose.pt')
modelo_m3.train(
    data=DATA_YAML_M3,
    epochs=80,
    imgsz=640,
    batch=16,
    device=0 if DEVICE == 'cuda' else 'cpu',
    project='/content/runs/pose',
    name='m3_llantas',
    exist_ok=True,
    plots=True
)

val_m3 = YOLO(RUTA_PESOS_M3).val(data=DATA_YAML_M3, split='val')
df_m3 = pd.DataFrame([{
    'Modelo': 'M3 - YOLO11s-pose (Ruedas/Ejes)',
    'Box mAP@50': round(float(val_m3.box.map50), 4),
    'Pose mAP@50 (OKS)': round(float(val_m3.pose.map50), 4),
    'Pose mAP@50-95 (OKS)': round(float(val_m3.pose.map), 4),
    'Precisión Pose': round(float(val_m3.pose.mp), 4),
    'Recall Pose': round(float(val_m3.pose.mr), 4)
}])
display(df_m3)

## 5. Evaluación del Error en el Conteo de Ejes sobre el conjunto de validación
Compara el número real de ejes (según las etiquetas de validación) frente al número de ejes estimado por M3 para reportar el **MAE (Error Absoluto Medio)** y el **% de Exactitud en Ejes** en el Informe Técnico.

In [ ]:
def evaluar_error_conteo_ejes(dir_val_imgs, dir_val_labels, ruta_modelo=RUTA_PESOS_M3, conf=0.30):
    m = YOLO(ruta_modelo)
    registros = []
    for ruta_img in sorted(glob.glob(os.path.join(dir_val_imgs, '*.*'))):
        nombre = os.path.splitext(os.path.basename(ruta_img))[0]
        ruta_lbl = os.path.join(dir_val_labels, nombre + '.txt')
        if not os.path.exists(ruta_lbl):
            continue
        lineas = [l.strip() for l in open(ruta_lbl).read().splitlines() if l.strip()]
        ejes_reales = max(2, len(lineas))

        im = cv2.imread(ruta_img)
        r = m.predict(im, conf=conf, device=DEVICE, verbose=False)[0]
        llantas_pred = []
        if r.keypoints is not None and len(r.boxes) > 0:
            xy = r.keypoints.xy.cpu().numpy()
            for b, s, pts in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), xy):
                cx, cy = pts[0][:2] if len(pts) > 0 else ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)
                llantas_pred.append({'box': b.tolist(), 'conf': float(s), 'centro': (float(cx), float(cy))})
        ejes_pred, _ = estimar_ejes_desde_llantas(llantas_pred)
        registros.append({
            'imagen': os.path.basename(ruta_img),
            'ejes_reales': ejes_reales,
            'ejes_estimados': ejes_pred,
            'error_abs': abs(ejes_reales - ejes_pred),
            'acierto': int(ejes_reales == ejes_pred)
        })
    df_ejes = pd.DataFrame(registros)
    if len(df_ejes) > 0:
        print(f"MAE Conteo de Ejes: {df_ejes['error_abs'].mean():.3f} ejes | Exactitud Exacta: {df_ejes['acierto'].mean()*100:.1f}%")
    return df_ejes

## 6. Validación de M1 (Detección de Placas + EasyOCR + Post-procesamiento Boliviano)
Sube las fotos de `Placas_en_mi_caso` para medir la precisión del OCR con normalización de placas bolivianas (`3 o 4 dígitos + 3 letras`, p. ej. `3034DUS`, `1620AEN`, `4222TLN`, `5314PFC`, `4694INB`, `1857NTE`, `4763FSS`, `1651FUL`, `4454FGX`, `4695ELB`, `1605KLX`, `1367ZTG`, `1205HGL`, `1318IZH`, `1506UAX`, `1152SFU`).

In [ ]:
import easyocr

lector_ocr = easyocr.Reader(['es', 'en'], gpu=(DEVICE == 'cuda'))

# Mapa de corrección posicional para formato de placa boliviana (3-4 números seguidos de 3 letras)
LETRA_A_NUM = {'O': '0', 'Q': '0', 'D': '0', 'I': '1', 'L': '1', 'Z': '2', 'S': '5', 'G': '6', 'B': '8'}
NUM_A_LETRA = {'0': 'O', '1': 'I', '2': 'Z', '5': 'S', '6': 'G', '8': 'B'}
PALABRAS_IGNORAR = {'BOLIVIA', 'NISSAN', 'TOYOTA', 'JEEP', 'MANTENGA', 'DISTANCIA', 'GRAND', 'CHEROKEE', 'HILUX'}

def normalizar_placa_bolivia(texto_crudo: str) -> str:
    limpio = re.sub(r'[^A-Z0-9]', '', texto_crudo.upper())
    for palabra in PALABRAS_IGNORAR:
        limpio = limpio.replace(palabra, '')
    # Buscar patrón de 4 dígitos/confusiones + 3 letras/confusiones (7 caracteres) o 3 dígitos + 3 letras (6 caracteres)
    m7 = re.search(r'([0-9OQDI LZSGB]{4})([A-Z012568]{3})', limpio.replace(' ', ''))
    if m7:
        p_num = ''.join(LETRA_A_NUM.get(c, c) for c in m7.group(1))
        p_let = ''.join(NUM_A_LETRA.get(c, c) for c in m7.group(2))
        if p_num.isdigit() and p_let.isalpha():
            return p_num + p_let
    m6 = re.search(r'([0-9OQDILZSGB]{3})([A-Z012568]{3})', limpio)
    if m6:
        p_num = ''.join(LETRA_A_NUM.get(c, c) for c in m6.group(1))
        p_let = ''.join(NUM_A_LETRA.get(c, c) for c in m6.group(2))
        if p_num.isdigit() and p_let.isalpha():
            return p_num + p_let
    return limpio

def extraer_placa_m1(img_bgr):
    # Preprocesamiento con mejora de contraste CLAHE para placas nocturnas o con reflejo
    gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gris)
    resultados = lector_ocr.readtext(clahe)
    candidatos = []
    for bbox, txt, conf in resultados:
        txt_up = re.sub(r'[^A-Z0-9]', '', txt.upper())
        if not txt_up or txt_up in PALABRAS_IGNORAR or len(txt_up) < 4:
            continue
        norm = normalizar_placa_bolivia(txt_up)
        es_formato_bo = bool(re.match(r'^\d{3,4}[A-Z]{3}$', norm))
        xs = [int(p[0]) for p in bbox]
        ys = [int(p[1]) for p in bbox]
        caja = [min(xs), min(ys), max(xs), max(ys)]
        candidatos.append({
            'texto_leido': txt,
            'texto': norm,
            'confianza': round(float(conf), 3),
            'bbox': caja,
            'prioridad': (1 if es_formato_bo else 0, float(conf), len(norm))
        })
    if not candidatos:
        return {'texto': '', 'texto_leido': '', 'confianza': 0.0, 'bbox': [0, 0, 0, 0]}
    mejor = max(candidatos, key=lambda c: c['prioridad'])
    return {k: mejor[k] for k in ('texto', 'texto_leido', 'confianza', 'bbox')}

print('Prueba unitaria de normalización boliviana:', normalizar_placa_bolivia('BOLIVIA 3034-DUS'), normalizar_placa_bolivia('162OAEN'))